# META-CXR — Table 6 BERTScore với **MedGemma-1.5-4b** (encoder toggling)

Phiên bản MedGemma của `meta-cxr-table6.ipynb`. Giữ **nguyên** phương pháp của
Table 6: một checkpoint `07_all_three` duy nhất, **toggle** các stream encoder
bên trong MHCAC + Q-Former, rồi sinh phần *Findings* và tính **BERTScore**
(deberta-xlarge-mnli) trên một subset test. Khác biệt duy nhất so với notebook
gốc: bộ sinh báo cáo **Vicuna-7B → MedGemma-1.5-4b + LoRA**, ảnh được đưa vào
LLM qua **Q-Former embedding injection** (32 `<image_soft_token>`).

| Paper label | Encoder trong repo |
|-------------|--------------------|
| RN50 | BioViL-T (CNN backbone) |
| ViT  | PubMedCLIP |
| Swin | Swin Transformer |

## ⚠ Lưu ý quan trọng (đọc trước khi diễn giải kết quả)

MedGemma LoRA của Google **không** được huấn luyện chung với projection
`img_proj_layer` (768 → hidden) của META-CXR, nên lớp projection này được
**khởi tạo ngẫu nhiên** lúc eval (giống `eval_bertscore_medgemma_qformer.py`).
Vì vậy con số BERTScore ở đây **minh hoạ pipeline**, **không** so sánh trực tiếp
được với giá trị Vicuna trong paper. Cột `Paper BERTScore` chỉ để tham khảo.

## Yêu cầu (Kaggle)

- **Accelerator:** GPU **T4 ×2** (hoặc A100). MedGemma-4b + toàn bộ stack vision
  đông cứng (BioViL-T + PubMedCLIP + Swin) khó nằm vừa 1 GPU 16GB → notebook đặt
  meta-cxr trên `cuda:1`, MedGemma trên `cuda:0`. Internet **ON**.
- **Kaggle Secrets:**
  - `GCS_SERVICE_ACCOUNT` — service-account JSON (hoặc base64) có quyền đọc
    `gs://meta-cxr-checkpoint` (chứa `07_all_three/checkpoint_best.pth`).
  - `HF_TOKEN` — Hugging Face token có quyền truy cập MedGemma (model gated).
- **3 datasets** gắn dưới `/kaggle/input/datasets/phuong20052/`:
  `mimic-cxr-jpg-lite`, `mimic-cxr-reported`, `mimic-cxr-p10-processed`.

Chạy lần lượt từ trên xuống.

## Cell 0 — Load Kaggle Secrets (`GCS_SERVICE_ACCOUNT` + `HF_TOKEN`)

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

os.environ["GCS_SERVICE_ACCOUNT"] = user_secrets.get_secret("GCS_SERVICE_ACCOUNT")
print("Kaggle secret loaded: GCS_SERVICE_ACCOUNT")

# MedGemma is a gated model -> a valid HF token is required to download weights.
hf_token = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
print("Kaggle secret loaded: HF_TOKEN")


## Cell 1 — Install Dependencies

Bộ dependency dùng `transformers>=4.53` (bắt buộc cho Gemma 3 / MedGemma) cùng
`peft` / `accelerate` / `torchao` / `sentencepiece`. Đây là môi trường đã được
`eval_bertscore_medgemma_qformer.py` dùng để load đồng thời cả stack META-CXR
Q-Former lẫn MedGemma. **Khác** notebook Vicuna (vốn pin `transformers==4.44.2`).

In [ ]:
"""
Cell 1 — Install dependencies (Kaggle).

Chạy cell này đầu tiên sau khi Restart Kernel. Cell này giữ numpy và
pandas ở major version 2 để tương thích với Kaggle/Python mới.
"""
import shutil
import subprocess
import sys
from pathlib import Path



def pip_install(*packages):
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--upgrade",
            "--upgrade-strategy", "only-if-needed", *packages,
        ],
        check=True,
    )


PACKAGES = [
    # OpenCV 4.12 requires numpy>=2,<2.3 on Python >=3.9.
    "opencv-python>=4.12,<4.13",
    "scikit-image>=0.22",
    "scikit-learn>=1.4",
    "omegaconf==2.3.0",
    "pycocoevalcap",
    "torchinfo",
    "wandb",
    "loralib==0.1.1",
    "iterative-stratification",
    "iopath",
    "hi-ml-multimodal==0.2.2",
    "timm>=0.9.0",
    "spacy>=3.8,<3.9",
    "nltk>=3.9",
    "google-cloud-storage",
    # Gemma 3 / MedGemma checkpoints need Transformers with `gemma3` support.
    "transformers>=4.53.0,<5",
    "tokenizers>=0.21,<0.22",
    # accelerate / safetensors / torchao / peft: required by transformers 4.53+
    # for Gemma 3 / MedGemma model loading with device_map + LoRA.
    "accelerate>=0.34",
    "safetensors>=0.5",
    "torchao>=0.16.0",
    "peft>=0.14.0",
    "bert-score>=0.3.13",
    "sentencepiece",
]
pip_install(*PACKAGES)
pip_install(
    "https://github.com/explosion/spacy-models/releases/download/"
    "en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"
)

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

java_bin = shutil.which("java")
JAVA_HOME_DETECTED = (
    str(Path(java_bin).resolve().parent.parent)
    if java_bin
    else "/usr/lib/jvm/java-11-openjdk-amd64"
)
print(f"Detected JAVA_HOME: {JAVA_HOME_DETECTED}")

import cv2 as _cv2
import numpy as _np
import pandas as _pd
import torch as _torch
import transformers as _transformers
from packaging.version import Version as _Version

if _Version(_transformers.__version__) < _Version("4.53.0"):
    raise RuntimeError(
        f"transformers phải >= 4.53.0 để load Gemma 3/MedGemma, "
        f"hiện tại là {_transformers.__version__}. "
        "Nếu vừa upgrade trong kernel cũ, hãy Restart Kernel/Session rồi chạy lại từ đầu."
    )

if not _np.__version__.startswith("2."):
    raise RuntimeError(f"numpy phải là major version 2, hiện tại là {_np.__version__}")
if not _pd.__version__.startswith("2."):
    raise RuntimeError(f"pandas phải là major version 2, hiện tại là {_pd.__version__}")

print(f"numpy   = {_np.__version__}")
print(f"pandas  = {_pd.__version__}")
print(f"opencv  = {_cv2.__version__}")
print(f"torch   = {_torch.__version__}")
print(f"transformers = {_transformers.__version__}")
print(f"GPUs available: {_torch.cuda.device_count()}")
for _i in range(_torch.cuda.device_count()):
    print(f"  GPU {_i}: {_torch.cuda.get_device_name(_i)}")


## Cell 2 — Clone Repository

In [ ]:
import os

REPO_DIR = "/kaggle/working/META-CXR"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/minhphuong150505/Meta-CXR-Kaggle.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Cell 3 — Verify Kaggle Datasets & Stage Reports CSV

Kiểm tra 3 dataset đã gắn và copy `mimic_cxr_cleaned.csv` về `/kaggle/working/`
(đường dẫn mà dataset loader mong đợi).

In [ ]:
import os
import shutil
import pandas as pd

KAGGLE_INPUT   = "/kaggle/input/datasets/phuong20052/mimic-cxr-jpg-lite"
IMAGE_ROOT     = KAGGLE_INPUT
REPORTS_ROOT   = "/kaggle/input/datasets/phuong20052/mimic-cxr-reported"
PROCESSED_ROOT = "/kaggle/input/datasets/phuong20052/mimic-cxr-p10-processed"

REQUIRED_CSVS = [
    "mimic-cxr-2.0.0-split.csv",
    "mimic-cxr-2.0.0-chexpert.csv",
    "mimic-cxr-2.0.0-metadata.csv",
]
PROCESSED_REQUIRED = ["train.csv", "val.csv", "test.csv"]
CLEANED_CSV   = "mimic_cxr_cleaned.csv"
REPORTS_LOCAL = "/kaggle/working/mimic_cxr_cleaned.csv"

for name, path in {"KAGGLE_INPUT": KAGGLE_INPUT, "REPORTS_ROOT": REPORTS_ROOT,
                   "PROCESSED_ROOT": PROCESSED_ROOT}.items():
    if not os.path.isdir(path):
        raise FileNotFoundError(f"{name} not found: {path}")

for fname in REQUIRED_CSVS:
    if not os.path.exists(os.path.join(KAGGLE_INPUT, fname)):
        raise FileNotFoundError(f"Missing metadata CSV: {fname}")
for fname in PROCESSED_REQUIRED:
    if not os.path.exists(os.path.join(PROCESSED_ROOT, fname)):
        raise FileNotFoundError(f"Missing preprocessed CSV: {fname}")

csv_in_dataset = os.path.join(REPORTS_ROOT, CLEANED_CSV)
if not os.path.exists(csv_in_dataset):
    raise FileNotFoundError(f"{CLEANED_CSV} not found at {csv_in_dataset}")
os.makedirs(os.path.dirname(REPORTS_LOCAL), exist_ok=True)
if not os.path.exists(REPORTS_LOCAL) or os.path.getsize(REPORTS_LOCAL) != os.path.getsize(csv_in_dataset):
    shutil.copy2(csv_in_dataset, REPORTS_LOCAL)

for k, v in {
    "KAGGLE_INPUT": KAGGLE_INPUT, "IMAGE_ROOT": IMAGE_ROOT,
    "REPORTS_ROOT": REPORTS_ROOT, "PROCESSED_ROOT": PROCESSED_ROOT,
    "REPORTS_CSV": REPORTS_LOCAL,
}.items():
    os.environ[k] = v

print("All datasets present.")
print(f"Reports CSV: {REPORTS_LOCAL} ({len(pd.read_csv(REPORTS_LOCAL))} rows)")

## Cell 4 — Write `configs/env_config.yaml`

In [ ]:
import os
import subprocess

result = subprocess.run("readlink -f $(which java) | sed 's|/bin/java||'",
                        shell=True, capture_output=True, text=True)
java_home = result.stdout.strip() or "/usr/lib/jvm/java-8-openjdk-amd64/jre"
java_path = java_home + "/bin:"

KAGGLE_INPUT   = os.environ["KAGGLE_INPUT"]
IMAGE_ROOT     = os.environ["IMAGE_ROOT"]
REPORTS_CSV    = os.environ["REPORTS_CSV"]
PROCESSED_ROOT = os.environ["PROCESSED_ROOT"]

env_config_content = f"""paths:
  data_root: \"{KAGGLE_INPUT}\"
  mimic_cxr_jpg_root: \"{IMAGE_ROOT}\"
  split_csv: \"{KAGGLE_INPUT}/mimic-cxr-2.0.0-split.csv\"
  reports_csv: \"{REPORTS_CSV}\"
  chexpert_csv: \"{KAGGLE_INPUT}/mimic-cxr-2.0.0-chexpert.csv\"
  metadata_csv: \"{KAGGLE_INPUT}/mimic-cxr-2.0.0-metadata.csv\"
  processed_dir: \"{PROCESSED_ROOT}\"
  processed_train_csv: \"{PROCESSED_ROOT}/train.csv\"
  processed_val_csv: \"{PROCESSED_ROOT}/val.csv\"
  processed_test_csv: \"{PROCESSED_ROOT}/test.csv\"
  output_dir: \"/kaggle/temp/output\"
  checkpoint_dir: \"/kaggle/temp/checkpoints\"
  gcs_bucket: \"gs://meta-cxr-checkpoint\"
  gcs_project: \"mimic-cxr-jpg-491409\"

wandb:
  entity: \"phuongnm150505-uit\"
  project: \"meta-cxr-encoder-comparison\"

java:
  home: \"{java_home}\"
  path: \"{java_path}\"
"""

os.makedirs("configs", exist_ok=True)
with open("configs/env_config.yaml", "w") as f:
    f.write(env_config_content)
print(env_config_content)

## Cell 5 — Download `07_all_three/checkpoint_best.pth` from GCS

Tải checkpoint all-encoders từ `gs://meta-cxr-checkpoint/07_all_three/` bằng
secret `GCS_SERVICE_ACCOUNT`. Hard-fail nếu checkpoint không tồn tại.

In [ ]:
import base64
import json
import os
from pathlib import Path

GCS_PROJECT = "mimic-cxr-jpg-491409"
GCS_BUCKET  = "meta-cxr-checkpoint"
GCS_PREFIX  = "07_all_three"
CKPT_NAME   = "checkpoint_best.pth"
LOCAL_CKPT  = Path("/kaggle/temp/checkpoints/07_all_three/checkpoint_best.pth")


def _load_service_account_info():
    raw = os.environ.get("GCS_SERVICE_ACCOUNT")
    if not raw:
        raise RuntimeError("GCS_SERVICE_ACCOUNT secret not set (run Cell 0).")
    raw = raw.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return json.loads(base64.b64decode(raw).decode("utf-8"))


from google.cloud import storage
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_info(_load_service_account_info())
client = storage.Client(project=GCS_PROJECT, credentials=credentials)

blob = client.bucket(GCS_BUCKET).blob(f"{GCS_PREFIX}/{CKPT_NAME}")
if not blob.exists():
    raise FileNotFoundError(
        f"gs://{GCS_BUCKET}/{GCS_PREFIX}/{CKPT_NAME} not found. "
        "Train/upload 07_all_three first, or check the service-account bucket access."
    )

LOCAL_CKPT.parent.mkdir(parents=True, exist_ok=True)
blob.download_to_filename(str(LOCAL_CKPT))
size_mb = LOCAL_CKPT.stat().st_size / (1024 ** 2)
print(f"Downloaded gs://{GCS_BUCKET}/{GCS_PREFIX}/{CKPT_NAME} -> {LOCAL_CKPT} ({size_mb:.1f} MB)")

## Cell 6 — MedGemma config

Điền `MEDGEMMA_MODEL_ID` và `MEDGEMMA_LORA_ID` (để `''` nếu không dùng LoRA).
Giá trị này được truyền cho evaluator ở Cell 8 qua CLI argument.

In [ ]:
# === MedGemma model / LoRA (sửa nếu cần) ===
MEDGEMMA_MODEL_ID = "google/medgemma-1.5-4b-it"
MEDGEMMA_LORA_ID  = "DeepRadiology/medgemma1.5-CXR"   # "" nếu không dùng LoRA

# Generation / eval params
MAX_NEW_TOKENS    = 300
TEST_SAMPLE_LIMIT = 300   # số sample test; tăng/giảm theo thời gian session

print("MEDGEMMA_MODEL_ID =", MEDGEMMA_MODEL_ID)
print("MEDGEMMA_LORA_ID  =", MEDGEMMA_LORA_ID or "(none)")
print("MAX_NEW_TOKENS    =", MAX_NEW_TOKENS)
print("TEST_SAMPLE_LIMIT =", TEST_SAMPLE_LIMIT)


## Cell 7 — Write the Table 6 (MedGemma) Evaluator

Ghi `eval_encoder_toggle_table6_medgemma.py`. **Toggle logic** lấy từ Vicuna
evaluator (`eval_encoder_toggle_table6.py`): cùng checkpoint `07_all_three`, mask
các image stream feeding cả MHCAC findings lẫn Q-Former tokens. **Generation +
scoring** tái sử dụng `MedGemmaQFormerGenerator` / `classify_with_thresholds` /
`build_prompt` / `compute_bertscore` từ `evaluation/eval_bertscore_medgemma_qformer.py`
(inject 32 Q-Former embeddings vào MedGemma qua `<image_soft_token>`).

In [ ]:
%%writefile eval_encoder_toggle_table6_medgemma.py
#!/usr/bin/env python3
"""Table 6 (MedGemma): report-generation BERTScore under encoder toggling.

Mirrors evaluation/eval_encoder_toggle_table6.py (the Vicuna Table 6 evaluator):
loads the single ``07_all_three`` checkpoint and, for each encoder configuration,
masks the image streams that feed BOTH the MHCAC findings and the Q-Former
generation tokens -- only the selected encoders pass through. The Findings
section is generated with MedGemma + LoRA via Q-Former embedding injection
(reusing MedGemmaQFormerGenerator from eval_bertscore_medgemma_qformer.py) and
scored with BERTScore.

NOTE: MedGemma's LoRA was not trained jointly with META-CXR's Q-Former
projection, so the 768 -> hidden img_proj_layer is randomly initialized at
evaluation time. The BERTScore numbers illustrate the pipeline; they are NOT
directly comparable to the paper's Vicuna values.
"""

from __future__ import annotations

import argparse
import gc
import json
import random
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

import model.lavis.tasks as tasks
from model.lavis.common.config import Config
from model.lavis.common.registry import registry

# Registration imports required by the LAVIS registry.
from model.lavis.common.optims import LinearWarmupCosineLRScheduler, LinearWarmupStepLRScheduler  # noqa: F401
from model.lavis.datasets.builders import *  # noqa: F401,F403
from model.lavis.models import *  # noqa: F401,F403
from model.lavis.processors import *  # noqa: F401,F403
from model.lavis.tasks import *  # noqa: F401,F403
from model.lavis.data.ReportDataset import MIMIC_CXR_Dataset
from local_config import VIS_ROOT

# Generation + scoring reused from the canonical MedGemma Q-Former evaluator.
# Importing this module is side-effect-safe (its main() is __main__-guarded);
# it provides the MedGemma generator plus the prompt/classify/score helpers that
# are identical to the Vicuna path except for the image-token block.
from evaluation.eval_bertscore_medgemma_qformer import (
    MedGemmaQFormerGenerator,
    classify_with_thresholds,
    build_prompt,
    compute_bertscore,
    filter_state_dict_for_model,
    NUM_IMG_TOKENS,
    MEDGEMMA_MODEL_ID,
    MEDGEMMA_LORA_ID,
)

registry.mapping["paths"]["cache_root"] = "."

SEED = 16

# Encoder configurations (paper Table 6), same toggle table as the Vicuna script.
TOGGLE_RUNS = [
    {"run": "01_biovil_only",       "RN50": True,  "ViT": False, "Swin": False},
    {"run": "02_pubmedclip_only",   "RN50": False, "ViT": True,  "Swin": False},
    {"run": "03_swin_only",         "RN50": False, "ViT": False, "Swin": True},
    {"run": "04_biovil_pubmedclip", "RN50": True,  "ViT": True,  "Swin": False},
    {"run": "05_biovil_swin",       "RN50": True,  "ViT": False, "Swin": True},
    {"run": "06_pubmedclip_swin",   "RN50": False, "ViT": True,  "Swin": True},
    {"run": "07_all_three",         "RN50": True,  "ViT": True,  "Swin": True},
]

# Paper Table 6 BERTScore (Vicuna pipeline). Reference only -- MedGemma + random
# projection makes the absolute delta meaningless; only relative trends matter.
PAPER_BERTSCORE = {
    "01_biovil_only": 0.312,
    "02_pubmedclip_only": 0.289,
    "03_swin_only": 0.267,
    "04_biovil_pubmedclip": 0.401,
    "05_biovil_swin": 0.394,
    "06_pubmedclip_swin": None,
    "07_all_three": 0.426,
}

# raddino is disabled for 07_all_three, so its newly-initialized alignment head
# is never used at inference and is legitimately absent from the checkpoint.
# The two Qformer.* prefixes cover the Q-Former MLM head + word embeddings, which
# are 1 token wider in the checkpoint (vocab 30523, trained under transformers
# 4.44.2 with the [DEC] bos token) than in a model built under transformers 4.53
# (vocab 30522). Those tensors are dropped by filter_state_dict_for_model and are
# never used at inference (encode_toggled passes query_embeds + image features
# only, never text token ids or the MLM head), so treat them as allowed-missing.
ALLOWED_MISSING_PREFIXES = (
    "visual_encoder.",
    "pubmedclip.model.",
    "swin.model.",
    "mhcac.embedding_alignment.raddino_",
    "Qformer.cls.predictions.",
    "Qformer.bert.embeddings.word_embeddings.",
)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--project-dir", type=Path, default=Path(__file__).resolve().parent)
    parser.add_argument(
        "--cfg",
        type=Path,
        default=Path("pretraining/configs/encoder_comparison/07_all_three.yaml"),
    )
    parser.add_argument(
        "--checkpoint",
        type=Path,
        default=Path("/kaggle/temp/checkpoints/07_all_three/checkpoint_best.pth"),
    )
    parser.add_argument("--limit", type=int, default=300, help="Subsample N test samples.")
    parser.add_argument("--batch-size", type=int, default=1)
    parser.add_argument("--num-workers", type=int, default=2)
    parser.add_argument("--max-new-tokens", type=int, default=300)
    parser.add_argument("--medgemma-model-id", default=MEDGEMMA_MODEL_ID)
    parser.add_argument("--medgemma-lora-id", default=MEDGEMMA_LORA_ID)
    parser.add_argument("--device-map-auto", action="store_true")
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--out-dir", type=Path, default=Path("output/encoder_toggle_table6_medgemma"))
    parser.add_argument("--allow-frozen-missing", action=argparse.BooleanOptionalAction, default=True)
    return parser.parse_args()


def seed_everything(device: str) -> None:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if device.startswith("cuda"):
        torch.cuda.manual_seed_all(SEED)


def load_torch_checkpoint(path: Path):
    try:
        return torch.load(str(path), map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(str(path), map_location="cpu")


def build_cfg(cfg_path: Path) -> Config:
    args = SimpleNamespace(cfg_path=str(cfg_path), options=None)
    return Config(args)


def validate_load_result(missing, unexpected, allow_frozen_missing) -> None:
    if unexpected:
        raise RuntimeError(f"Unexpected checkpoint keys: {unexpected[:20]}")
    if allow_frozen_missing:
        invalid = [k for k in missing if not k.startswith(ALLOWED_MISSING_PREFIXES)]
        if invalid:
            raise RuntimeError(f"Checkpoint missing non-frozen/non-backbone keys: {invalid[:20]}")
    elif missing:
        raise RuntimeError(f"Checkpoint is missing keys: {missing[:20]}")


def build_model(cfg: Config, checkpoint_path: Path, device: str, allow_frozen_missing: bool):
    task = tasks.setup_task(cfg)
    model = task.build_model(cfg)
    ckpt = load_torch_checkpoint(checkpoint_path)
    state_dict = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    # Drop shape-mismatched tensors (e.g. the Q-Former MLM head / word embeddings
    # that are 1 token wider in the 4.44.2-trained checkpoint than in a 4.53-built
    # model). They are never used at inference; see ALLOWED_MISSING_PREFIXES.
    state_dict, mismatched = filter_state_dict_for_model(model, state_dict)
    # assign=True is required under transformers >= 4.53: init_Qformer builds the
    # Q-Former via BertLMHeadModel.from_pretrained(..., add_cross_attention=True),
    # whose low-cpu-mem/meta init leaves the cross-attention params (present in
    # the checkpoint) on the META device. A plain copy_ (assign=False) is a no-op
    # on meta tensors, so they stay on meta and model.to(device) later raises
    # "Cannot copy out of meta tensor". assign=True swaps in the checkpoint
    # tensors, materializing those params with real data.
    missing, unexpected = model.load_state_dict(state_dict, strict=False, assign=True)
    print(
        f"Loaded {checkpoint_path}; missing={len(missing)}, "
        f"unexpected={len(unexpected)}, mismatched_skipped={len(mismatched)}"
    )
    for key, ckpt_shape, model_shape in mismatched[:5]:
        print(f"  skipped shape mismatch: {key}: checkpoint={ckpt_shape}, model={model_shape}")
    validate_load_result(list(missing), list(unexpected), allow_frozen_missing)

    # Safety net: any param/buffer still on meta after the assign-load would crash
    # model.to(device) with a cryptic error. Surface it explicitly with the names
    # so the cause is actionable (it would mean a meta tensor that is absent from
    # the checkpoint, e.g. a backbone that failed to materialize at init).
    meta_params = [n for n, p in model.named_parameters() if getattr(p, "is_meta", False)]
    meta_buffers = [n for n, b in model.named_buffers() if getattr(b, "is_meta", False)]
    if meta_params or meta_buffers:
        raise RuntimeError(
            "Tensors still on the meta device after load_state_dict(assign=True): "
            f"params={meta_params[:10]} (total {len(meta_params)}), "
            f"buffers={meta_buffers[:10]} (total {len(meta_buffers)}). "
            "These keys are not in the checkpoint; they must be materialized at "
            "model-init time (rebuild the affected backbone without low_cpu_mem "
            "meta init) before this evaluator can place the model on the GPU."
        )

    # assign=True adopts the checkpoint tensors' dtype; the original (copy_-based)
    # pipeline ran the meta-cxr stack in fp32. Normalize to fp32 so the toggled
    # encoders, image batch, and Q-Former all share one dtype (encode_toggled
    # also .float()s its outputs). .float() only casts floating-point tensors, so
    # integer buffers (e.g. position_ids) are left intact.
    model.float()
    model.to(device)
    # Pubmedclip hardcodes self.device='cuda' (=cuda:0) at init and uses it to
    # place its inputs in forward; realign with the actual device so it works
    # when the meta-cxr model is placed on cuda:1.
    if getattr(model, "pubmedclip", None) is not None:
        model.pubmedclip.device = device
    model.eval()
    return model


def make_test_loader(cfg: Config, batch_size: int, num_workers: int, limit: int | None, device: str):
    dataset = MIMIC_CXR_Dataset(
        vis_processor=None,
        text_processor=None,
        vis_root=VIS_ROOT,
        split="test",
        cfg=cfg,
        truncate=None,
    )
    if limit is not None:
        dataset = Subset(dataset, list(range(min(limit, len(dataset)))))
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=device.startswith("cuda"),
    )


@torch.no_grad()
def encode_toggled(model, image, use_rn50: bool, use_vit: bool, use_swin: bool):
    """Replicate forward_image with per-encoder masking.

    Only the selected encoders' Q-Former projections are concatenated and fed to
    the Q-Former; the same selection masks the MHCAC streams. Returns toggled
    (classification_logits, qformer_embeds) on CPU.
    """
    streams = []
    cnn_patches = vit_patches = swin_patches = None

    if model.use_biovil:
        cnn_raw = model.visual_encoder(image).projected_patch_embeddings.reshape(
            image.shape[0], -1, 1408
        )
        cnn_patches = model.ln_vision(cnn_raw)
        if use_rn50:
            streams.append(cnn_patches)

    if model.use_pubmedclip:
        vit_patches, pubmed_projection = model.pubmedclip(image, apply_aug=False)
        if use_vit:
            streams.append(pubmed_projection)

    if model.use_swin:
        swin_patches = model.swin(image)
        if use_swin:
            streams.append(model.swin_qformer_proj(swin_patches))

    if not streams:
        raise ValueError("No encoder stream selected for this configuration.")

    concat_image_embeds = torch.cat(streams, dim=1)

    cls_logits, _, _, _, _ = model.mhcac(
        cnn_patches=cnn_patches if use_rn50 else None,
        vit_patches=vit_patches if use_vit else None,
        swin_patches=swin_patches if use_swin else None,
        raddino_patches=None,
        text_embeddings=None,
        labels=None,
    )

    image_atts = torch.ones(concat_image_embeds.size()[:-1], dtype=torch.long, device=image.device)
    query_tokens = model.query_tokens.expand(concat_image_embeds.shape[0], -1, -1)
    query_output = model.Qformer.bert(
        query_embeds=query_tokens,
        encoder_hidden_states=concat_image_embeds,
        encoder_attention_mask=image_atts,
        return_dict=True,
    )
    return cls_logits.float().cpu(), query_output.last_hidden_state.float().cpu()


def main() -> None:
    args = parse_args()
    project_dir = args.project_dir.resolve()
    cfg_path = args.cfg if args.cfg.is_absolute() else project_dir / args.cfg
    checkpoint_path = args.checkpoint if args.checkpoint.is_absolute() else project_dir / args.checkpoint
    out_dir = args.out_dir if args.out_dir.is_absolute() else project_dir / args.out_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    seed_everything(args.device)

    print("project_dir =", project_dir)
    print("cfg_path    =", cfg_path)
    print("checkpoint  =", checkpoint_path)
    print("MedGemma    =", args.medgemma_model_id)
    print("LoRA        =", args.medgemma_lora_id or "(none)")
    print("limit       =", args.limit, "| max_new_tokens =", args.max_new_tokens)

    # MedGemma is loaded whole on cuda:0; on 2+ GPUs put the meta-cxr encoder/
    # Q-Former model on cuda:1 to avoid co-location OOM. Its Q-Former outputs are
    # moved to CPU in encode_toggled, so the LLM device is independent.
    meta_device = args.device
    if args.device == "cuda" and torch.cuda.device_count() >= 2:
        meta_device = "cuda:1"
    print("meta_device =", meta_device, "| MedGemma on cuda:0")

    cfg = build_cfg(cfg_path)
    model = build_model(cfg, checkpoint_path, meta_device, args.allow_frozen_missing)
    loader = make_test_loader(cfg, args.batch_size, args.num_workers, args.limit, meta_device)
    n_samples = len(loader.dataset)
    print("test samples =", n_samples)

    # Fast fail: exercise the Q-Former forward path BEFORE the slow MedGemma
    # load so a transformers<->Qformer.py incompatibility surfaces in seconds.
    # The repo pins transformers==4.44.2 for Qformer.py elsewhere; this notebook
    # runs >=4.53 (required by MedGemma), so verify the toggle path runs here.
    probe_batch = next(iter(loader))
    probe_image = probe_batch["image"].to(meta_device, non_blocking=True)
    _probe_logits, probe_embs = encode_toggled(model, probe_image, True, True, True)
    assert tuple(probe_embs.shape[1:]) == (NUM_IMG_TOKENS, 768), (
        f"Q-Former output {tuple(probe_embs.shape)} != (B, {NUM_IMG_TOKENS}, 768)"
    )
    print(f"Q-Former forward OK (probe embs {tuple(probe_embs.shape)})")
    del probe_batch, probe_image, _probe_logits, probe_embs

    # Load MedGemma once; reused across all 7 toggled configurations.
    generator = MedGemmaQFormerGenerator(
        args.medgemma_model_id,
        args.medgemma_lora_id,
        device_map_auto=args.device_map_auto,
    )

    rows = []
    details = {}
    for item in TOGGLE_RUNS:
        run_name = item["run"]
        print(f"\n{'='*60}\n{run_name}\n{'='*60}")
        predictions, references = [], []
        for batch in tqdm(loader, desc=f"{run_name} gen"):
            image = batch["image"].to(meta_device, non_blocking=True)
            cls_logits, qformer_embs = encode_toggled(
                model, image, item["RN50"], item["ViT"], item["Swin"]
            )
            for i in range(cls_logits.shape[0]):
                groups = classify_with_thresholds(cls_logits[i])
                prompt = build_prompt(groups)
                try:
                    pred = generator.generate(prompt, qformer_embs[i], args.max_new_tokens)
                except Exception as exc:  # noqa: BLE001 - keep going, record empty pred
                    print(f"  generate failed for sample {len(predictions)}: {exc}")
                    pred = ""
                ref_field = batch["text_output"]
                ref = ref_field[i] if isinstance(ref_field, (list, tuple)) else str(ref_field[i])
                predictions.append(pred)
                references.append(str(ref))

        mean_f1 = compute_bertscore(predictions, references)
        paper = PAPER_BERTSCORE[run_name]
        rows.append(
            {
                "Run": run_name,
                "RN50": "yes" if item["RN50"] else "no",
                "ViT": "yes" if item["ViT"] else "no",
                "Swin": "yes" if item["Swin"] else "no",
                "BERTScore": round(mean_f1, 4),
                "N_samples": len(predictions),
                "Paper BERTScore": paper if paper is not None else None,
                "Delta vs Paper": round(mean_f1 - paper, 4) if paper is not None else None,
            }
        )
        details[run_name] = {"bertscore_f1": mean_f1, "n_samples": len(predictions)}

        out_jsonl = out_dir / f"reports_{run_name}.jsonl"
        with out_jsonl.open("w", encoding="utf-8") as f:
            for p, r in zip(predictions, references):
                f.write(json.dumps({"pred": p, "ref": r}) + "\n")
        print(f">>> {run_name}: BERTScore F1={mean_f1:.4f} ({len(predictions)} samples) -> {out_jsonl}")

    table = pd.DataFrame(rows)
    table_path = out_dir / "table6_bertscore_medgemma_table.csv"
    json_path = out_dir / "table6_bertscore_medgemma_details.json"
    table.to_csv(table_path, index=False)
    with json_path.open("w", encoding="utf-8") as f:
        json.dump(
            {
                "checkpoint": str(checkpoint_path),
                "cfg": str(cfg_path),
                "limit": args.limit,
                "num_samples": n_samples,
                "llm": args.medgemma_model_id,
                "lora": args.medgemma_lora_id,
                "image_conditioning": "Q-Former embeddings injected at 32 <image_soft_token> positions",
                "img_projection": "random Linear(768, llm_hidden); MedGemma LoRA not co-trained with it",
                "max_new_tokens": args.max_new_tokens,
                "details": details,
            },
            f,
            indent=2,
        )

    print("\nTable 6 (MedGemma):")
    print(table.to_string(index=False))
    print("\nWrote:", table_path)
    print("Wrote:", json_path)

    del model, loader
    gc.collect()
    if args.device.startswith("cuda"):
        torch.cuda.empty_cache()


if __name__ == "__main__":
    main()


## Cell 8 — Run the Evaluation

Một lượt qua subset MIMIC-CXR test; encoder stream được toggle cho từng cấu hình,
MedGemma load một lần và tái dùng. Ghi `output/encoder_toggle_table6_medgemma/*`.

In [ ]:
# Build the command in Python (f-string) so the MedGemma config values are
# substituted reliably, then run it. IPython's {var} expansion inside a
# multi-line `!` command is unreliable, so we expand here and run `!{cmd}`.
cmd = (
    "cd /kaggle/working/META-CXR && python eval_encoder_toggle_table6_medgemma.py"
    " --project-dir /kaggle/working/META-CXR"
    " --checkpoint /kaggle/temp/checkpoints/07_all_three/checkpoint_best.pth"
    f" --medgemma-model-id {MEDGEMMA_MODEL_ID}"
    f" --medgemma-lora-id '{MEDGEMMA_LORA_ID}'"
    f" --max-new-tokens {MAX_NEW_TOKENS}"
    f" --limit {TEST_SAMPLE_LIMIT}"
)
print(cmd)
# Stream output directly so the real traceback is visible.
!{cmd}


## Cell 9 — Table 6 (MedGemma)

Đọc output của evaluator và render bảng. `BERTScore` là giá trị run này;
`Paper BERTScore` là giá trị Vicuna trong paper (chỉ tham khảo); `Delta vs Paper`
là chênh lệch (không mang ý nghĩa so sánh trực tiếp — xem lưu ý đầu notebook).

In [ ]:
import pandas as pd
df = pd.read_csv(
    "/kaggle/working/META-CXR/output/encoder_toggle_table6_medgemma/table6_bertscore_medgemma_table.csv"
)
df
